In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")  # adjust if your notebook isn't inside notebooks/

orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
geolocation = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
category_translation = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

print("Loaded all 9 tables")

Loaded all 9 tables


In [2]:
tables = {
    "orders": orders,
    "order_items": order_items,
    "customers": customers,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "geolocation": geolocation,
    "category_translation": category_translation,
}

for name, df in tables.items():
    print(f"{name}: {df.shape}")

orders: (99441, 8)
order_items: (112650, 7)
customers: (99441, 5)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
geolocation: (1000163, 5)
category_translation: (71, 2)


In [3]:
orders.isna().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [4]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [5]:
print("Unique customer_id:", customers["customer_id"].nunique())
print("Unique customer_unique_id:", customers["customer_unique_id"].nunique())

Unique customer_id: 99441
Unique customer_unique_id: 96096


In [6]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [7]:
delivered_orders = orders[orders["order_status"] == "delivered"].copy()
print(f"Filtered from {len(orders)} to {len(delivered_orders)} delivered orders")

Filtered from 99441 to 96478 delivered orders


In [11]:
order_items_agg = order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    avg_item_price=("price", "mean"),
).reset_index()

order_items_agg.head(10)

,order_id,n_items,total_price,total_freight,avg_item_price
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,58.90
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,239.90
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,199.00
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,12.99
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,199.90
5,00048cc3ae777c65dbb7d2a0634bc1ea,1,21.90,12.69,21.90
6,00054e8431b9d7675808bcb819fb4a32,1,19.90,11.85,19.90
7,000576fe39319847cbb9d288c5617fa6,1,810.00,70.75,810.00
8,0005a1a1728c9d785b8e2b08b904576c,1,145.95,11.65,145.95
9,0005f50442cb953dcd1d21e1fb923495,1,53.99,11.40,53.99


In [ ]:
payments_agg = payments.groupby("order_id").agg(
    n_payment_methods=("payment_type", "nunique"),
    total_payment_value=("payment_value", "sum"),
    max_installments=("payment_installments", "max"),
).reset_index()

# capture the most common/primary payment type separately
primary_payment_type = (
    payments.sort_values("payment_value", ascending=False)
    .drop_duplicates(subset="order_id", keep="first")[["order_id", "payment_type"]]
    .rename(columns={"payment_type": "primary_payment_type"})
)

payments_agg = payments_agg.merge(primary_payment_type, on="order_id", how="left")
payments_agg.head()